In [0]:
# Import sys and custom functions
import sys
import posixpath
import uuid, time
from pyspark.sql.functions import col, lit
from src.pipeline_config import *
from src.schema_builder import build_schema

In [0]:
# Databricks parameters
dbutils.widgets.text("dataset", "") 
dbutils.widgets.text("layer", "")

dataset = dbutils.widgets.get("dataset").strip()
layer = dbutils.widgets.get("layer").strip()

validate_parameters(dataset, layer)

#Load yaml and validate
yaml_cfg = load_yaml(dataset, layer)
validate_yaml(yaml_cfg, layer)

In [0]:
#Create run id
run_id = str(uuid.uuid4())

#Create variables
catalog = yaml_cfg['target']['catalog']
schema = yaml_cfg['target']['schema']
target_table = f"{catalog}.{schema}.{yaml_cfg['target']['table']}"
file_format = yaml_cfg["file"]["format"]
options = build_options(yaml_cfg["file"])
ingest_ts = spark.sql("select current_timestamp() as ts").collect()[0]["ts"]

# Create paths
landing_root = f"/Volumes/{catalog}/landing"
file_path = f"{landing_root}/files/{dataset}/"
checkpoint_path = f"{landing_root}/checkpoints/{dataset}/"
schema_path = f"{landing_root}/checkpoints/schema/{dataset}/"

In [0]:
print(f"[PARAMS] dataset={dataset} catalog={catalog} layer={layer} run_id={run_id}")
print(f"[CTX] target_table={target_table}")
print(f"[CTX] schema_path={schema_path}")
print(f"[CTX] file_path={file_path}")
print(f"[CTX] checkpoint_path={checkpoint_path}")
print(f"[CTX] ingest_ts={ingest_ts}")

start = time.time()
print(f"[BRONZE][START] run_id={run_id} dataset={dataset}")

#Autoloader
try:
    df = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", file_format)
            .option("cloudFiles.schemaLocation", schema_path)
            .options(**options)
            .load(file_path)
            .withColumn("_ingest_ts", lit(ingest_ts))  # Timestamp of ingestion
            .withColumn("_source_file_path", col("_metadata.file_path"))  # Full path
            .withColumn("_source_mod_time", col("_metadata.file_modification_time"))  # Last modification timestamp of the source file in storage
            .withColumn("_source_size", col("_metadata.file_size"))  # Size of the source file in bytes
            .writeStream
            .outputMode("append")
            .format("delta")
            .option("checkpointLocation", checkpoint_path)
            .trigger(availableNow=True)
            .toTable(target_table)
    )

    duration_s = round(time.time() - start, 3)
    print(f"[BRONZE][SUCCESS] run_id={run_id} duration_s={duration_s}")
except Exception as e:
    duration_s = round(time.time() - start, 3)
    print(f"[BRONZE][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise